# 01 — Agent Loop dari Nol: Reason → Act → Observe

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riki-profile/ai-engineer-journey/blob/main/11-ai-agent/01_agent_loop_dari_nol.ipynb)

Di modul 08 kita memakai **tool use**: model meminta fungsi dijalankan, kita menjalankannya, lalu mengirim hasilnya. Sebuah **AI agent** adalah perluasan pola itu: LLM yang bekerja dalam **loop**, memutuskan sendiri langkah berikutnya berdasarkan hasil langkah sebelumnya, sampai tugasnya selesai.

```
          ┌──────────────────────────────────────────────┐
tugas ──▶ │ REASON: model memikirkan langkah berikutnya  │
          │ ACT:    model memanggil tool (SQL, kalkulator)│ ◀─┐
          │ OBSERVE: kode kita menjalankan tool,         │   │ ulangi sampai
          │          hasilnya dikirim kembali ke model   │ ──┘ model selesai
          └──────────────────────────────────────────────┘
```

Studi kasus: **agen analis data** untuk database penjualan toko fiktif *Toko Elektronik Nusantara* (lanjutan modul 09). Agen menerima pertanyaan bisnis dalam bahasa Indonesia, menulis SQL sendiri, memperbaiki query yang error, lalu menjawab. Kamu sudah paham SQL, jadi kamu bisa menilai sendiri apakah langkah agen masuk akal.

Isi notebook:
1. Database toko dan tool-tool agen.
2. Loop agen dari nol, dengan jejak (*trace*) setiap langkah.
3. Pemulihan dari error.
4. Workflow vs agent: kapan loop bebas justru tidak perlu.
5. Biaya dan jumlah langkah.

> Biaya API notebook ini diperkirakan kurang dari US$1.

## 0. Instalasi, API Key, dan Database

Seperti modul 08: simpan `ANTHROPIC_API_KEY` di **Colab Secrets** (ikon 🔑, aktifkan *Notebook access*) atau di file `.env` (salin dari `.env.example` di root repo). **Jangan menulis API key di kode.** Notebook ini tidak butuh GPU.

In [ ]:
%pip install -q "anthropic>=1.9" python-dotenv pandas pydantic

In [ ]:
import ast
import json
import operator
import os
import sqlite3
import urllib.request
from pathlib import Path

import anthropic
import pandas as pd
from dotenv import find_dotenv, load_dotenv

QUICK_RUN = os.environ.get("QUICK_RUN") == "1"

MODEL = "claude-opus-5-5"
# Refusal fallback (beta): jika permintaan ditolak, server mencoba ulang di model cadangan (lihat modul 08)
FALLBACK = {"betas": ["server-side-fallback-2026-07-01"], "fallbacks": "default"}


def ambil_api_key():
    """Cari API key: environment variable / file .env (lokal), lalu Colab Secrets."""
    load_dotenv(find_dotenv(usecwd=True))
    if os.getenv("ANTHROPIC_API_KEY"):
        return os.getenv("ANTHROPIC_API_KEY")
    try:
        from google.colab import userdata
        return userdata.get("ANTHROPIC_API_KEY")
    except Exception:  # bukan di Colab, atau Secret belum dibuat / belum diberi akses
        return None


api_key = ambil_api_key()
if not api_key:
    raise RuntimeError(
        "ANTHROPIC_API_KEY belum diatur.\n"
        "- Colab: ikon 🔑 Secrets → tambah ANTHROPIC_API_KEY → aktifkan Notebook access, lalu jalankan ulang sel ini.\n"
        "- Lokal: salin .env.example menjadi .env di root repo, lalu isi ANTHROPIC_API_KEY.")
client = anthropic.Anthropic(api_key=api_key)
print(f"✅ API key ditemukan | model: {MODEL}")

In [ ]:
# File pendukung modul ini (ada di repo; di Colab diunduh dari GitHub)
REPO_RAW = "https://raw.githubusercontent.com/riki-profile/ai-engineer-journey/main/11-ai-agent"
for nama_file in ["toko_db.py", "server_toko.py"]:
    if not Path(nama_file).exists():
        urllib.request.urlretrieve(f"{REPO_RAW}/{nama_file}", nama_file)

from toko_db import buat_database  # noqa: E402  (diimpor setelah file tersedia)

DB = buat_database("toko.db")
print(f"Database: {DB} ({DB.stat().st_size / 1e6:.1f} MB)")

## 1. Database Toko

Database dibuat oleh [`toko_db.py`](toko_db.py) dengan data acak (seed tetap): 18 produk, 300 pelanggan, 3.000 pesanan (Januari 2025 – Juni 2026), item pesanan, dan ulasan. Semua fiktif. Kita intip dulu dengan pandas + SQL, seperti yang biasa kamu lakukan.

In [ ]:
with sqlite3.connect(DB) as con:
    for tabel in ["produk", "pelanggan", "pesanan", "item_pesanan", "ulasan"]:
        n = con.execute(f"SELECT COUNT(*) FROM {tabel}").fetchone()[0]
        print(f"{tabel:13}: {n:>5} baris")
    display(pd.read_sql("SELECT * FROM pesanan LIMIT 3", con))
    display(pd.read_sql("""
        SELECT strftime('%Y-%m', p.tanggal) AS bulan, SUM(i.jumlah * i.harga_satuan) AS pendapatan
        FROM pesanan p JOIN item_pesanan i ON i.pesanan_id = p.id
        WHERE p.status = 'selesai' GROUP BY bulan ORDER BY bulan LIMIT 4""", con))

## 2. Tool Agen

Tiga tool dengan tanggung jawab yang jelas:

| Tool | Fungsi | Pengaman |
|---|---|---|
| `lihat_skema` | Melihat struktur tabel | Baca-saja |
| `jalankan_sql` | Menjalankan query SELECT | Koneksi **read-only** (`mode=ro`), satu statement, maks 50 baris |
| `hitung` | Kalkulator (persentase, selisih) | Tanpa `eval`: hanya angka dan operator aritmetika |

Kalkulator terpisah itu penting: LLM bisa keliru saat berhitung "di kepala", terutama angka besar. Lebih andal meminta model menulis ekspresinya lalu kode kita yang menghitung.

Tool yang **dibatasi** lebih aman daripada tool serba bisa. `jalankan_sql` hanya bisa membaca; tool yang mengubah data (misalnya membatalkan pesanan) dibahas di notebook 03 dengan persetujuan manusia.

In [ ]:
def tool(nama, deskripsi, properti=None, wajib=None):
    """Buat definisi tool strict. Semua properti wajib kecuali `wajib` diisi."""
    properti = properti or {}
    return {"name": nama, "description": deskripsi, "strict": True,
            "input_schema": {"type": "object", "properties": properti,
                             "required": list(properti) if wajib is None else wajib, "additionalProperties": False}}


MAKS_BARIS = 50


def koneksi_baca_saja():
    return sqlite3.connect(f"{DB.resolve().as_uri()}?mode=ro", uri=True)


def lihat_skema():
    with koneksi_baca_saja() as con:
        return "\n\n".join(r[0] for r in con.execute("SELECT sql FROM sqlite_master WHERE type = 'table'"))


def jalankan_sql(query):
    con = koneksi_baca_saja()
    try:
        cur = con.execute(query)
        kolom = [d[0] for d in cur.description or []]
        baris = cur.fetchmany(MAKS_BARIS + 1)
    finally:
        con.close()
    return {"kolom": kolom, "baris": [list(b) for b in baris[:MAKS_BARIS]], "terpotong": len(baris) > MAKS_BARIS}


TOOL_SKEMA = tool("lihat_skema", "Tampilkan perintah CREATE TABLE semua tabel (kolom, tipe, dan komentar arti kolom). "
                                 "Panggil ini dulu jika belum tahu struktur database.")
TOOL_SQL = tool("jalankan_sql", "Jalankan SATU query SQLite SELECT (baca-saja) pada database toko dan kembalikan maksimal "
                                "50 baris. Gunakan agregasi (SUM, COUNT, AVG, GROUP BY) agar hasilnya ringkas.",
                {"query": {"type": "string", "description": "Satu query SQLite SELECT"}})


OPERATOR = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
            ast.Pow: operator.pow, ast.USub: operator.neg}


def hitung(ekspresi):
    """Kalkulator aman: hanya angka dan + - * / ** (tanpa eval)."""
    def nilai(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in OPERATOR:
            kiri, kanan = nilai(node.left), nilai(node.right)
            if isinstance(node.op, ast.Pow) and abs(kanan) > 100:
                raise ValueError("pangkat terlalu besar")
            return OPERATOR[type(node.op)](kiri, kanan)
        if isinstance(node, ast.UnaryOp) and type(node.op) in OPERATOR:
            return OPERATOR[type(node.op)](nilai(node.operand))
        raise ValueError("hanya angka dan operator + - * / ** yang diizinkan")
    return nilai(ast.parse(ekspresi, mode="eval").body)


TOOLS = [TOOL_SKEMA, TOOL_SQL,
         tool("hitung", "Hitung ekspresi aritmetika, misalnya pertumbuhan persen: (120 - 100) / 100 * 100. "
                        "Gunakan untuk semua perhitungan setelah mendapat angka dari SQL.",
              {"ekspresi": {"type": "string", "description": "Ekspresi dengan angka dan + - * / ** ( )"}})]
FUNGSI = {"lihat_skema": lihat_skema, "jalankan_sql": jalankan_sql, "hitung": hitung}

# Uji tool secara langsung sebelum diberikan ke agen
print(jalankan_sql("SELECT status, COUNT(*) FROM pesanan GROUP BY status"))
print(hitung("(1461687000 - 1208560000) / 1208560000 * 100"))

## 3. Loop Agen

Fungsi `jalankan_agen` di bawah adalah inti modul ini. Perhatikan:
- Riwayat `messages` terus bertambah: pesan user → respons model (termasuk blok `thinking` dan `tool_use`) → hasil tool → respons berikutnya, dan seterusnya.
- **Reason**: dengan `thinking={"type": "adaptive", "display": "summarized"}`, kita bisa melihat ringkasan pertimbangan model (💭) sebelum ia bertindak.
- **Act**: blok `tool_use`. Model bisa meminta **beberapa tool sekaligus** dalam satu langkah; semua dijalankan dan hasilnya dikirim bersama.
- **Observe**: hasil tool dikirim sebagai `tool_result`. Jika tool error, error-nya juga dikirim (`is_error`) agar model bisa memperbaiki langkahnya.
- **Kondisi berhenti**: model selesai (`stop_reason` bukan `tool_use`), atau batas `max_langkah` tercapai. Tanpa batas, agen yang "bingung" bisa berputar terus dan menghabiskan biaya.

In [ ]:
SYSTEM = """Kamu analis data Toko Elektronik Nusantara (TEN). Jawab pertanyaan bisnis dengan memeriksa database SQLite toko.
- Jika belum tahu strukturnya, lihat skema dulu. Pendapatan = SUM(jumlah * harga_satuan) dari pesanan berstatus 'selesai', kecuali diminta lain.
- Gunakan SQL untuk menghitung; jangan menebak angka.
- Jawaban akhir ringkas dalam bahasa Indonesia, sebutkan angka penting dalam rupiah dan asumsi yang kamu pakai.
- Gunakan tool hitung untuk semua perhitungan (persentase, selisih, rata-rata dari angka hasil query)."""

HARGA_PER_JUTA = {"input": 4.00, "output": 20.00}   # USD, claude-opus-5-5 (cek halaman harga resmi)


def biaya(usage):
    return (usage.input_tokens * HARGA_PER_JUTA["input"] + usage.output_tokens * HARGA_PER_JUTA["output"]) / 1e6


def teks(respon):
    return "".join(b.text for b in respon.content if b.type == "text")


def jalankan_agen(tugas, max_langkah=10, tampilkan=True):
    messages = [{"role": "user", "content": tugas}]
    jejak = []
    for langkah in range(1, max_langkah + 1):
        r = client.beta.messages.create(
            model=MODEL, max_tokens=16000, system=SYSTEM, tools=TOOLS, messages=messages,
            thinking={"type": "adaptive", "display": "summarized"}, **FALLBACK)
        messages.append({"role": "assistant", "content": r.content})          # simpan SELURUH blok respons
        panggilan = [b for b in r.content if b.type == "tool_use"]
        jejak.append({"langkah": langkah, "tool": ", ".join(b.name for b in panggilan) or "-",
                      "token_input": r.usage.input_tokens, "token_output": r.usage.output_tokens,
                      "biaya (US$)": round(biaya(r.usage), 4)})
        if tampilkan:
            print(f"── langkah {langkah} ──")
            for b in r.content:
                if b.type == "thinking" and b.thinking.strip():
                    print(f"💭 {b.thinking.strip()[:300]}")
                elif b.type == "text" and b.text.strip() and panggilan:
                    print(f"💬 {b.text.strip()[:300]}")

        if r.stop_reason != "tool_use":                                         # model selesai
            awalan = "" if r.stop_reason == "end_turn" else f"[berhenti: {r.stop_reason}] "
            return awalan + teks(r), pd.DataFrame(jejak), messages

        hasil = []
        for b in panggilan:                                                     # ACT
            try:
                keluaran = FUNGSI[b.name](**b.input)
                hasil.append({"type": "tool_result", "tool_use_id": b.id,
                              "content": json.dumps(keluaran, ensure_ascii=False, default=str)})
            except Exception as e:
                hasil.append({"type": "tool_result", "tool_use_id": b.id, "content": f"Error: {e}", "is_error": True})
            if tampilkan:                                                       # OBSERVE
                print(f"🔧 {b.name}({json.dumps(b.input, ensure_ascii=False)[:250]})")
                print(f"   👀 {'❌ ' if hasil[-1].get('is_error') else ''}{hasil[-1]['content'][:250]}")
        messages.append({"role": "user", "content": hasil})
    return "[berhenti: melebihi max_langkah]", pd.DataFrame(jejak), messages

Tugas pertama: pertanyaan sederhana yang butuh beberapa langkah (melihat skema, menulis query, menjawab).

In [ ]:
jawaban, jejak, riwayat = jalankan_agen("Berapa total pendapatan bulan Maret 2026?")
print(f"\n🤖 {jawaban}")
jejak

Cocokkan dengan query kita sendiri. Selalu verifikasi agen pada kasus yang jawabannya kamu tahu, sebelum memercayainya untuk pertanyaan yang jawabannya tidak kamu tahu.

In [ ]:
benar = jalankan_sql("""
    SELECT SUM(i.jumlah * i.harga_satuan) FROM pesanan p JOIN item_pesanan i ON i.pesanan_id = p.id
    WHERE p.status = 'selesai' AND p.tanggal BETWEEN '2026-03-01' AND '2026-03-31'""")["baris"][0][0]
print(f"Jawaban acuan: Rp{benar:,}".replace(",", "."))

Tugas yang lebih sulit: butuh beberapa query dan perhitungan. Perhatikan bagaimana agen memecah masalah sendiri.

In [ ]:
TUGAS = [
    "Bandingkan pendapatan kategori laptop pada kuartal 1 2025 dan kuartal 1 2026. Berapa persen pertumbuhannya?",
    "Di kota mana pelanggan tier Platinum paling banyak? Berapa rata-rata nilai per pesanan (selesai) "
    "dari pelanggan Platinum di kota itu?",
]
semua_jejak = {"Maret 2026": jejak}
for tugas in TUGAS[:1] if QUICK_RUN else TUGAS:
    print(f"\n🧑 {tugas}")
    jawaban, jejak_tugas, _ = jalankan_agen(tugas)
    print(f"\n🤖 {jawaban}")
    semua_jejak[tugas[:40]] = jejak_tugas

## 4. Pemulihan dari Error

Salah satu kelebihan agen dibanding satu panggilan LLM: agen bisa **melihat error dan mencoba lagi**. Jika model menebak nama kolom yang salah, SQLite mengembalikan error, lalu model membacanya dan memperbaiki query. Ini pesan yang akan dilihat model:

In [ ]:
for query in ["SELECT total_harga FROM pesanan LIMIT 1", "SELECT * FROM pesanan; DROP TABLE pesanan"]:
    try:
        jalankan_sql(query)
    except Exception as e:
        print(f"{query!r}\n   → Error: {e}")

Pesan error yang **jelas** membantu model memperbaiki diri. Saat merancang tool, tulis pesan error yang menjelaskan apa yang salah dan apa yang diharapkan (misalnya "kolom `total_harga` tidak ada; lihat skema").

Lihat juga jejak tugas-tugas di atas: apakah ada langkah dengan ❌ yang kemudian diperbaiki?

## 5. Workflow vs Agent

Tidak semua masalah butuh agen:
- **Workflow**: langkah-langkahnya **ditentukan kode** (misalnya: buat SQL → jalankan → rangkum). Lebih murah, cepat, dan mudah diprediksi.
- **Agent**: **model** yang menentukan langkah dan kapan selesai. Lebih fleksibel untuk tugas terbuka, tetapi lebih mahal dan lebih sulit diuji.

Mulailah dari yang paling sederhana. Berikut versi workflow *text-to-SQL* untuk pertanyaan yang sama: tepat 2 panggilan LLM, tanpa loop. Kelemahannya: jika query pertama salah, tidak ada kesempatan memperbaiki.

In [ ]:
from pydantic import BaseModel, Field


class RencanaSQL(BaseModel):
    query: str = Field(description="Satu query SQLite SELECT yang menjawab pertanyaan")


def workflow_sql(pertanyaan):
    total = 0.0
    # Langkah 1 (LLM): terjemahkan pertanyaan menjadi SQL, dengan skema di prompt
    r1 = client.beta.messages.parse(
        model=MODEL, max_tokens=16000, output_config={"effort": "low"}, output_format=RencanaSQL,
        system=f"Tulis satu query SQLite untuk menjawab pertanyaan.\n<skema>\n{lihat_skema()}\n</skema>\n"
               "Pendapatan = SUM(jumlah * harga_satuan) dari pesanan berstatus 'selesai'.",
        messages=[{"role": "user", "content": pertanyaan}], **FALLBACK)
    total += biaya(r1.usage)
    query = r1.parsed_output.query if r1.parsed_output else ""
    # Langkah 2 (kode): jalankan query
    try:
        data = jalankan_sql(query)
    except Exception as e:
        return f"[workflow gagal: query error: {e}]", query, total
    # Langkah 3 (LLM): rangkum hasil
    r2 = client.beta.messages.create(
        model=MODEL, max_tokens=16000, output_config={"effort": "low"},
        messages=[{"role": "user", "content": f"Pertanyaan: {pertanyaan}\nQuery: {query}\nHasil: {data}\n"
                                              "Jawab pertanyaan secara ringkas dalam bahasa Indonesia."}], **FALLBACK)
    total += biaya(r2.usage)
    return teks(r2), query, total


jawaban_wf, query_wf, biaya_wf = workflow_sql("Berapa total pendapatan bulan Maret 2026?")
print(f"SQL: {query_wf}\n🤖 {jawaban_wf}\nBiaya workflow: US${biaya_wf:.4f} | "
      f"biaya agen untuk pertanyaan yang sama: US${semua_jejak['Maret 2026']['biaya (US$)'].sum():.4f}")

Untuk pertanyaan sederhana, workflow biasanya lebih murah dan cukup. Agen unggul saat jumlah langkah tidak bisa ditentukan di awal: pertanyaan yang butuh beberapa query berurutan, eksplorasi data, atau pemulihan dari error.

## 6. Jejak: Langkah, Token, dan Biaya

Setiap langkah mengirim **seluruh riwayat** lagi (API stateless), sehingga token input naik di setiap langkah. Inilah yang membuat agen dengan banyak langkah menjadi mahal, dan alasan memori/konteks perlu dikelola (notebook 02).

In [ ]:
ringkasan = pd.DataFrame([{"tugas": nama, "langkah": len(j), "token input total": j["token_input"].sum(),
                           "token output total": j["token_output"].sum(), "biaya (US$)": round(j["biaya (US$)"].sum(), 4)}
                          for nama, j in semua_jejak.items()])
ringkasan

## Ringkasan

- **Agent** = LLM + tools + **loop** reason → act → observe, dengan kondisi berhenti yang jelas (`stop_reason`, `max_langkah`).
- Simpan **seluruh** blok respons ke riwayat; kirim hasil tool (termasuk error) sebagai `tool_result`.
- Rancang tool yang **sempit dan aman** (SQL read-only, kalkulator tanpa `eval`) dengan pesan error yang jelas.
- **Verifikasi** agen pada pertanyaan yang jawabannya kamu tahu.
- Mulai dari **workflow** jika langkahnya bisa ditentukan; pakai agen untuk tugas terbuka.
- Biaya agen tumbuh dengan jumlah langkah karena riwayat dikirim ulang setiap langkah.

## Latihan

1. **Tool baru.** Tambahkan tool `grafik_batang(judul, label, nilai)` yang menggambar grafik dengan matplotlib dan mengembalikan teks "grafik ditampilkan". Minta agen: *"Tampilkan grafik pendapatan per kategori tahun 2025."* Apakah agen memakai hasil SQL dengan benar sebagai input grafik?
2. **Efek effort.** Tambahkan parameter `effort` ke `jalankan_agen` (diteruskan sebagai `output_config={"effort": ...}`), lalu jalankan ulang tugas perbandingan laptop dengan `"low"` dan `"high"`. Bandingkan jumlah langkah, biaya, dan kebenaran jawaban (verifikasi dengan SQL sendiri).
3. **Workflow dengan perbaikan.** Ubah `workflow_sql` agar jika query error, pesan error dikirim kembali ke LLM untuk memperbaiki query (maksimal 2 kali). Apa bedanya dengan agen? Di titik mana workflow semacam ini mulai "menjadi" agen?